# Phase 0 Preparation — Dataset generation

Implements the datasets described in `notebooks_benzon/plan.md` (a design guide,
not code to transcribe — see `vonto/dataset/`'s own module docstrings for what
changed during implementation and why). Four datasets, each built through the same
`Dataset`/`Seed`/`Inquiry` interface:

- **Synonyms** — procedurally generated pairs, constrained to each keyword's *real*
  WordNet synonyms and ranked by word2vec similarity for a genuine "distance" gradient
  (every distance is a real synonym, not just a "nearby" word).
- **List elicitation** — one seed per sampled keyword and per sampled temperature
  (`T` temperatures each); the actual list only exists once a later phase sends the
  resulting `Inquiry` to a model.
- **Twenty questions** — every game is fully played *here*, at generation time (real
  model calls, checkpointed after each game so an interruption doesn't lose progress).
- **TriviaQA** — downloaded and loaded, with a swappable question phrasing
  (`DIRECT_QUESTION` vs `BINARY_JUDGMENT_QUESTION`).

Each dataset gets exactly two cells: a **generation** cell that always calls
`generate()` and overwrites `$PROJECT_ROOT/data/prepared/<name>.json` (no
skip-if-already-cached check — the recurring bug this notebook kept hitting was a
stale cache silently matching a since-changed shape/class and never regenerating),
followed by a **verification** cell that independently reads the saved file back via
`load_if_cached()` and checks it. Every cell in this notebook (generation or
verification, any dataset) is fully self-contained — it does its own imports and
setup — so any single cell can be run on its own, in any order, without first
running any other cell.

## Synonyms

`n` word pairs per relation tier, drawn directly from human-annotated lexical-relation
benchmarks instead of sampling a keyword and generating a related word for it:
**twin** (interchangeable -- EVALution/CogALexV's labeled Synonym pairs, further
filtered to require real word2vec similarity), **sibling** (same meaning, not
interchangeable -- K&H+N's co-hyponym "sibl" pairs), **relative** (similar meaning --
BLESS's hyper/mero/attri/event pairs). Pairs don't share a word across tiers -- equal
pairs, not one keyword's three relations. Needs word2vec vectors for the twin filter
only -- downloaded once via `huggingface_hub` (gensim itself can't run on this
environment's Python version; see `vonto.dataset.synonyms_dataset.load_word2vec_vectors`'s
docstring for why this reimplements the raw binary reader instead).

In [1]:
import json, pathlib, sys

ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "vonto").is_dir())
sys.path.insert(0, str(ROOT))

from vonto import config as cfg
from vonto import dataset as DP

CONFIG_PATH = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "config.json").exists()) / "config.json"
CONFIG = json.loads(CONFIG_PATH.read_text())
DATA_DIR = (pathlib.Path.cwd() / "out" / "prepared")
DATA_DIR.mkdir(parents=True, exist_ok=True)

word2vec_path = DP.download_word2vec_vectors()
print("word2vec vectors:", word2vec_path)

synonyms = DP.SynonymsDataset(word2vec_path, **CONFIG["synonyms"])
synonyms.generate()  # always regenerates -- never skipped by an existing cache

expected = 3 * synonyms.n
print(f"\n{len(synonyms.seeds)} seeds (at most {expected}; a relation tier may fall "
      f"short if its source benchmark doesn't have enough qualifying pairs, see any warnings above)")

by_relation = {}
for s in synonyms.seeds:
    by_relation.setdefault(s.relation, []).append(s.wordpair)
for relation, pairs in by_relation.items():
    print(f"\n{relation} ({len(pairs)}):")
    for pair in pairs[:8]:
        print(f"  {pair}")

print("\nsample inquiry:", synonyms.inquiry(synonyms.seeds[0]))

path = synonyms.save(DATA_DIR)
print(f"\nsaved {len(synonyms.seeds)} seeds -> {path} (overwritten)")


word2vec vectors: /home/stud_homes/s7846062/.cache/huggingface/hub/models--NathaNn1111--word2vec-google-news-negative-300-bin/snapshots/78856d4586b3a938134c9833d92139f2e056e369/GoogleNews-vectors-negative300.bin
[SynonymsDataset] dropped 378/879 'Synonym'-labeled pairs below word2vec similarity 0.25 (not genuinely interchangeable)

201 seeds (at most 201; a relation tier may fall short if its source benchmark doesn't have enough qualifying pairs, see any warnings above)

twin (67):
  ('old', 'past')
  ('full', 'complete')
  ('cloth', 'material')
  ('boss', 'leader')
  ('atmosphere', 'mood')
  ('unusual', 'strange')
  ('ascend', 'rise')
  ('light', 'pale')

sibling (67):
  ('lotus', 'pondweed')
  ('goosefoot', 'sedum')
  ('electric', 'roadster')
  ('locust', 'termite')
  ('rose', 'artemisia')
  ('coriander', 'lobelia')
  ('wisent', 'otter')
  ('vicuna', 'fawn')

relative (67):
  ('sieve', 'separate')
  ('alligator', 'aggressive')
  ('dishwasher', 'plate')
  ('pub', 'destroy')
  ('desk',

In [2]:
import json, pathlib, sys

ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "vonto").is_dir())
sys.path.insert(0, str(ROOT))

from vonto import config as cfg
from vonto import dataset as DP

CONFIG_PATH = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "config.json").exists()) / "config.json"
CONFIG = json.loads(CONFIG_PATH.read_text())
DATA_DIR = (pathlib.Path.cwd() / "out" / "prepared")
synonyms = DP.SynonymsDataset(None, **CONFIG["synonyms"])  # word2vec_path unused on the load-from-cache path
assert synonyms.load_if_cached(DATA_DIR), (
    f"no cache found for {synonyms.shape_tag()} -- run the generation cell above first"
)

expected = 3 * synonyms.n
print(f"loaded {len(synonyms.seeds)} seeds from cache ({synonyms.shape_tag()}.json) (at most {expected})")
for relation in ("twin", "sibling", "relative"):
    count = sum(1 for s in synonyms.seeds if s.relation == relation)
    print(f"  {relation}: {count}")
print("PASS  synonyms cache verified")


loaded 201 seeds from cache (Synonyms_n67.json) (at most 201)
  twin: 67
  sibling: 67
  relative: 67
PASS  synonyms cache verified


## List elicitation

One seed per sampled keyword *and* per sampled temperature (`T` temperatures each,
uniform in `[0, 0.8)`): a random sampling temperature and generation seed. No model
call happens here — the list itself only exists once `inquiry(seed)`'s prompt is
actually sent to a model, in a later phase. Multiple temperatures per keyword (rather
than one) is what lets a later temperature-vs-variety correlation control for the
keyword's own inherent variance instead of confounding the two.

In [3]:
import json, pathlib, sys

ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "vonto").is_dir())
sys.path.insert(0, str(ROOT))

from vonto import config as cfg
from vonto import dataset as DP

CONFIG_PATH = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "config.json").exists()) / "config.json"
CONFIG = json.loads(CONFIG_PATH.read_text())
DATA_DIR = (pathlib.Path.cwd() / "out" / "prepared")
DATA_DIR.mkdir(parents=True, exist_ok=True)

list_elicitation = DP.ListElicitationDataset(**CONFIG["list_elicitation"])
list_elicitation.generate()  # always regenerates -- never skipped by an existing cache

expected = len(DP.CATEGORY) * list_elicitation.n * list_elicitation.T
print(f"{len(list_elicitation.seeds)} seeds (expected {expected})")
assert len(list_elicitation.seeds) == expected
# sample() only guarantees no repeats *within* one category's own draw, not
# globally across all 10 -- two different categories can legitimately share a word.
print(f"{len({s.keyword for s in list_elicitation.seeds})} distinct keywords across all categories")

for s in list_elicitation.seeds[:5]:
    print(" ", s, "->", list_elicitation.inquiry(s).question)

path = list_elicitation.save(DATA_DIR)
print(f"\nsaved {len(list_elicitation.seeds)} seeds -> {path} (overwritten)")


/scratch/qi/env/lib/python3.14/site-packages/nltk/downloader.py:1076: UserWarning: NLTK will not authorize the non-private download directory '/home/stud_homes/s7846062/nltk_data': it (or an ancestor) is world- or group-writable, so another local user could plant files there. Choose a private location such as ~/nltk_data.
  for msg in self.incr_download(info_or_id, download_dir, force):


200 seeds (expected 200)
10 distinct keywords across all categories
  ListElicitationSeed(keyword='killer', temperature=0.5095693498571635, generation_seed=1097657232) -> Give me a list of 20 things, starting with killer.
  ListElicitationSeed(keyword='killer', temperature=0.03277881914895575, generation_seed=579362556) -> Give me a list of 20 things, starting with killer.
  ListElicitationSeed(keyword='killer', temperature=0.013222108422823276, generation_seed=376383645) -> Give me a list of 20 things, starting with killer.
  ListElicitationSeed(keyword='killer', temperature=0.7302044618221775, generation_seed=1746484540) -> Give me a list of 20 things, starting with killer.
  ListElicitationSeed(keyword='killer', temperature=0.4853086206137439, generation_seed=2084654370) -> Give me a list of 20 things, starting with killer.

saved 200 seeds -> /home/stud_homes/s7846062/fatass/home/thesis/experiment/code_morph/notebooks_benzon/out/prepared/ListElicitation_n1k20T20.json (overwritten)


In [4]:
import json, pathlib, sys

ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "vonto").is_dir())
sys.path.insert(0, str(ROOT))

from vonto import config as cfg
from vonto import dataset as DP

CONFIG_PATH = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "config.json").exists()) / "config.json"
CONFIG = json.loads(CONFIG_PATH.read_text())
DATA_DIR = (pathlib.Path.cwd() / "out" / "prepared")
list_elicitation = DP.ListElicitationDataset(**CONFIG["list_elicitation"])
assert list_elicitation.load_if_cached(DATA_DIR), (
    f"no cache found for {list_elicitation.shape_tag()} -- run the generation cell above first"
)

expected = len(DP.CATEGORY) * list_elicitation.n * list_elicitation.T
print(f"loaded {len(list_elicitation.seeds)} seeds from cache ({list_elicitation.shape_tag()}.json) "
      f"(expected {expected})")
assert len(list_elicitation.seeds) == expected
print(f"{len({s.keyword for s in list_elicitation.seeds})} distinct keywords across all categories")
print("PASS  list_elicitation cache verified")


loaded 200 seeds from cache (ListElicitation_n1k20T20.json) (expected 200)
10 distinct keywords across all categories
PASS  list_elicitation cache verified


## Twenty questions

Every game is **played in full here** (two model generations per turn -- a question,
then a batched Yes/No partition -- for `T` turns), not deferred to a later phase: a
game's own history is needed to ask its next question at all, so there's no
meaningful "unplayed seed" to hand off. Checkpointed to
`$PROJECT_ROOT/data/prepared/twenty_questions_checkpoint.json` after every game.

This is the one dataset where "always regenerate" would mean real, expensive GPU
work if taken literally per game -- so `generate()` always runs, but its own
internal checkpoint resume (distinct from this notebook's now-removed
skip-if-already-cached pattern) still means an unfinished run resumes rather than
replaying already-played games, and a shape change (`k`/`T`) still discards a
now-stale checkpoint automatically.

In [5]:
import json, pathlib, sys

ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "vonto").is_dir())
sys.path.insert(0, str(ROOT))

from vonto import config as cfg
from vonto import dataset as DP

CONFIG_PATH = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "config.json").exists()) / "config.json"
CONFIG = json.loads(CONFIG_PATH.read_text())
DATA_DIR = (pathlib.Path.cwd() / "out" / "prepared")
from vonto.models import load_model

DATA_DIR.mkdir(parents=True, exist_ok=True)

loaded = load_model("qwen")  # the one dataset here that needs a real model -- it plays real games

twenty_questions = DP.TwentyQuestionsDataset(
    loaded, **CONFIG["twenty_questions"],
    checkpoint_path=DATA_DIR / "twenty_questions_checkpoint.json",
)
twenty_questions.generate()  # always called -- resumes from its own mid-game checkpoint, doesn't
                              # replay already-finished games

expected = len(DP.CATEGORY) * twenty_questions.n_games
print(f"\n{len(twenty_questions.seeds)} seeds (expected {expected})")
assert len(twenty_questions.seeds) == expected
for game in twenty_questions.seeds:
    assert len(game.keywords) == twenty_questions.k
    assert game.secret in game.keywords
    assert len(game.history) == twenty_questions.T, "every game should be fully played"

sample_game = twenty_questions.seeds[0]
print(f"\nsample game: secret={sample_game.secret!r}, {len(sample_game.keywords)} keywords, "
      f"{len(sample_game.history)} turns played")
print(sample_game.history_str())

path = twenty_questions.save(DATA_DIR)
print(f"\nsaved {len(twenty_questions.seeds)} seeds -> {path} (overwritten)")


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

[TwentyQuestionsDataset] played 1/200 games (animal.n.01)
[TwentyQuestionsDataset] played 2/200 games (animal.n.01)
[TwentyQuestionsDataset] played 3/200 games (animal.n.01)
[TwentyQuestionsDataset] played 4/200 games (animal.n.01)
[TwentyQuestionsDataset] played 5/200 games (animal.n.01)
[TwentyQuestionsDataset] played 6/200 games (animal.n.01)
[TwentyQuestionsDataset] played 7/200 games (animal.n.01)
[TwentyQuestionsDataset] played 8/200 games (animal.n.01)
[TwentyQuestionsDataset] played 9/200 games (animal.n.01)
[TwentyQuestionsDataset] played 10/200 games (animal.n.01)
[TwentyQuestionsDataset] played 11/200 games (animal.n.01)
[TwentyQuestionsDataset] played 12/200 games (animal.n.01)
[TwentyQuestionsDataset] played 13/200 games (animal.n.01)
[TwentyQuestionsDataset] played 14/200 games (animal.n.01)
[TwentyQuestionsDataset] played 15/200 games (animal.n.01)
[TwentyQuestionsDataset] played 16/200 games (animal.n.01)
[TwentyQuestionsDataset] played 17/200 games (animal.n.01)
[Twent

In [6]:
import json, pathlib, sys

ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "vonto").is_dir())
sys.path.insert(0, str(ROOT))

from vonto import config as cfg
from vonto import dataset as DP

CONFIG_PATH = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "config.json").exists()) / "config.json"
CONFIG = json.loads(CONFIG_PATH.read_text())
DATA_DIR = (pathlib.Path.cwd() / "out" / "prepared")
twenty_questions = DP.TwentyQuestionsDataset(
    None, **CONFIG["twenty_questions"],  # loaded model unused on the load-from-cache path
    checkpoint_path=DATA_DIR / "twenty_questions_checkpoint.json",
)
assert twenty_questions.load_if_cached(DATA_DIR), (
    f"no cache found for {twenty_questions.shape_tag()} -- run the generation cell above first"
)

expected = len(DP.CATEGORY) * twenty_questions.n_games
print(f"loaded {len(twenty_questions.seeds)} seeds from cache ({twenty_questions.shape_tag()}.json) "
      f"(expected {expected})")
assert len(twenty_questions.seeds) == expected
for game in twenty_questions.seeds:
    assert len(game.keywords) == twenty_questions.k
    assert game.secret in game.keywords
    assert len(game.history) == twenty_questions.T
print("PASS  twenty_questions cache verified")


loaded 200 seeds from cache (TwentyQuestions_n_games20k128T2.json) (expected 200)
PASS  twenty_questions cache verified


## TriviaQA

Downloaded-and-loaded (`vonto.data.load_triviaqa`), with the shared, hardcoded phrasing
variants: `DIRECT_QUESTION` asks the question as-is; `BINARY_JUDGMENT_QUESTION` asks
whether a candidate answer is right, yes/no — the same seed can be turned into either,
by swapping `raw_question`.

In [7]:
import json, pathlib, sys

ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "vonto").is_dir())
sys.path.insert(0, str(ROOT))

from vonto import config as cfg
from vonto import dataset as DP

CONFIG_PATH = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "config.json").exists()) / "config.json"
CONFIG = json.loads(CONFIG_PATH.read_text())
DATA_DIR = (pathlib.Path.cwd() / "out" / "prepared")
DATA_DIR.mkdir(parents=True, exist_ok=True)

triviaqa = DP.TriviaQA(**CONFIG["triviaqa"])
triviaqa.generate()  # always regenerates -- never skipped by an existing cache

print(f"{len(triviaqa.seeds)} seeds (expected {triviaqa.limit})")
assert len(triviaqa.seeds) == triviaqa.limit
assert len({s.trivia_question for s in triviaqa.seeds}) == len(triviaqa.seeds), "questions should be deduplicated"

sample_seed = triviaqa.seeds[0]
print(f"\nsample seed: {sample_seed}")
print("DIRECT_QUESTION       :", triviaqa.inquiry(sample_seed).question)
triviaqa.raw_question = DP.BINARY_JUDGMENT_QUESTION
print("BINARY_JUDGMENT_QUESTION:", triviaqa.inquiry(sample_seed).question)
triviaqa.raw_question = DP.DIRECT_QUESTION  # restore the default before saving

path = triviaqa.save(DATA_DIR)
print(f"\nsaved {len(triviaqa.seeds)} seeds -> {path} (overwritten)")


200 seeds (expected 200)

sample seed: TriviaQASeed(trivia_question='Who was the man behind The Chipmunks?', answer='David Seville', aliases=('David Seville',))
DIRECT_QUESTION       : Who was the man behind The Chipmunks?
BINARY_JUDGMENT_QUESTION: Is the answer to "Who was the man behind The Chipmunks?" "David Seville"? Answer Yes or No.

saved 200 seeds -> /home/stud_homes/s7846062/fatass/home/thesis/experiment/code_morph/notebooks_benzon/out/prepared/TriviaQA_limit200.json (overwritten)


In [8]:
import json, pathlib, sys

ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "vonto").is_dir())
sys.path.insert(0, str(ROOT))

from vonto import config as cfg
from vonto import dataset as DP

CONFIG_PATH = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "config.json").exists()) / "config.json"
CONFIG = json.loads(CONFIG_PATH.read_text())
DATA_DIR = (pathlib.Path.cwd() / "out" / "prepared")
triviaqa = DP.TriviaQA(**CONFIG["triviaqa"])
assert triviaqa.load_if_cached(DATA_DIR), (
    f"no cache found for {triviaqa.shape_tag()} -- run the generation cell above first"
)

print(f"loaded {len(triviaqa.seeds)} seeds from cache ({triviaqa.shape_tag()}.json) (expected {triviaqa.limit})")
assert len(triviaqa.seeds) == triviaqa.limit
assert len({s.trivia_question for s in triviaqa.seeds}) == len(triviaqa.seeds)
print("PASS  triviaqa cache verified")


loaded 200 seeds from cache (TriviaQA_limit200.json) (expected 200)
PASS  triviaqa cache verified
